# Conquer Mobile → TripoSR: اختبار أول مجسم
هذا الإصدار يبدأ بصورة شخصية واحدة فقط من الصور الأصلية. لا يبدأ معالجة الصور الـ42 إلا بعد نجاح الاختبار.

In [ ]:
import os, sys, subprocess, pathlib, shutil
from pathlib import Path
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'GPU غير متصل. اختَر T4 GPU ثم أعد تشغيل هذه الخلية.'
print('GPU:', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0).upper() or 'TESLA' in torch.cuda.get_device_name(0).upper(), 'تأكد أن نوع GPU هو T4/Tesla.'


In [ ]:
from pathlib import Path
REPO = Path('/content/Conquer-Mobile-Assets')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git',str(REPO)],check=True)
IMAGES = REPO / 'original_images' / 'characters'
image_files = sorted([p for p in IMAGES.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg','.webp')])
assert image_files, f'لم أجد صور الشخصيات في {IMAGES}'
TEST_IMAGE = image_files[0]
print('صورة الاختبار الأصلية:', TEST_IMAGE.name)
from PIL import Image
display(Image.open(TEST_IMAGE))


In [ ]:
TSR_REPO = Path('/content/TripoSR')
if not TSR_REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(TSR_REPO)],check=True)
os.chdir(TSR_REPO)
subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip','setuptools','wheel','scikit-build-core>=1.0','cmake','ninja','pybind11'],check=True)
# Install the official project dependencies after upgrading the CMake build backend.
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
print('Dependencies installed successfully.')


In [ ]:
runpy = TSR_REPO / 'run.py'
src = runpy.read_text()
old_line = 'meshes = model.extract_mesh(scene_codes, not args.bake_texture, resolution=args.mc_resolution)'
assert old_line in src, 'تغيّر كود TripoSR الرسمي؛ أوقفت التعديل بدل إجراء تعديل غير مضمون.'
src = src.replace(old_line,'meshes = model.extract_mesh(scene_codes, False, resolution=args.mc_resolution)',1)
old_export = 'meshes[0].export(out_mesh_path)'
new_export = 'meshes[0].export(out_mesh_path)\n        alt_format = "glb" if args.model_save_format == "obj" else "obj"\n        meshes[0].export(os.path.join(output_dir, str(i), f"mesh.{alt_format}"))'
assert old_export in src, 'لم أجد موضع تصدير المجسم المتوقع.'
src = src.replace(old_export,new_export,1)
runpy.write_text(src)
print('تم تجهيز التصدير بدون vertex colors وبصيغتي OBJ وGLB.')


In [ ]:
OUT = Path('/content/Conquer_TripoSR_First_Model')
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)
cmd = [sys.executable,str(runpy),str(TEST_IMAGE),'--output-dir',str(OUT/'raw'),'--model-save-format','obj','--mc-resolution','256','--chunk-size','4096']
subprocess.run(cmd,cwd=str(TSR_REPO),check=True)
generated = OUT/'raw'/'0'
obj_src, glb_src = generated/'mesh.obj', generated/'mesh.glb'
assert obj_src.exists() and glb_src.exists(), 'لم يتم إنشاء ملفي OBJ وGLB.'
safe = TEST_IMAGE.stem.replace(' ','_')
obj_dst, glb_dst = OUT/f'{safe}.obj', OUT/f'{safe}.glb'
shutil.copy2(obj_src,obj_dst); shutil.copy2(glb_src,glb_dst)
assert obj_dst.stat().st_size > 1000 and glb_dst.stat().st_size > 1000, 'ملفات التصدير فارغة أو غير مكتملة.'
print('نجاح أول تصدير!')
print('OBJ:',obj_dst,obj_dst.stat().st_size,'bytes')
print('GLB:',glb_dst,glb_dst.stat().st_size,'bytes')
from google.colab import files
files.download(str(glb_dst))
